## Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv('Challenge_3_100k_data.csv')

In [ ]:
import seaborn as sns
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.show()

In [ ]:
# Calculate the percentage of null values in each column
null_percentage = df.isnull().mean() * 100
print(null_percentage)

# Drop rows with null values
df = df.dropna()
df.describe(include=[np.float64])

In [ ]:
df['Density'] = df['Mp']/(df['Rp']**3*4/3*np.pi)
df['Mp_Me'] = df['Mp']/(5.9722*10**24)
df['Rp_Re'] = df['Rp']/(6371.000*10**3)

# Save the cleaned dataframe to a new CSV file
df.to_csv('cleaned_Challenge_3_100k_data.csv', index=False)
df.hist(figsize=(15, 10), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns 
sns.pairplot(df)

---------------

In [ ]:
features = df[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i', 'Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]
target = df[['Mp_Me', 'Rp_Re', 'Tp', 'C/O','logZ']]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier, VotingClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance
from sklearn.model_selection import learning_curve
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error#, root_mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold


def score(ytest, predict):
    r2 = r2_score(ytest, predict)
    MAE = mean_absolute_error(ytest, predict)
    MSE = mean_squared_error(ytest, predict)
    RMSE = np.sqrt(mean_squared_error(ytest, predict))
    print(f"R2 = {r2} \nMAE = {MAE} \nMSE = {MSE} \nRMSE = {RMSE}")
    return r2

## radius

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

X = df[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i', 'Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]
Y = df['Rp_Re'] #df[['Mp', 'Rp', 'Tp', 'C/O','logZ']]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define the features and target variables
X1 = X[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i', 'Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]
X1_train = X_train[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i', 'Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]
X1_test = X_test[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i', 'Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]
X2 = X[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i']]
X2_train = X_train[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i']]
X2_test = X1_test[['Bessel u', 'Bessel b', 'Bessel v', 'Bessel r', 'Bessel i']]
X3 = X[['Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]
X3_train = X_train[['Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]
X3_test = X_test[['Sloan u', 'Sloan g', 'Sloan r', 'Sloan i', 'Sloan z']]

In [ ]:
r_2_for_Rp_Re = []

for i in range(1, 4):  # Best Filter Use
    if i == 1:
        print("Use Both Filter")
    elif i == 2:
        print("Use Bessel Filter")
    elif i == 3:
        print("Use Sloan Filter")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))  # Create a 2x2 grid for subplots
    r_2_for_each_i = []
    
    for n in range(1, 5):  # Test degrees from 1 to 4
        print(f"\nModel with polyFeature = {n}")
        
        # Define parameter grid for fine-tuning
        param_grid = {
            'polyFeature__degree': [n],  # Set polynomial degree
            'scaler': [StandardScaler(), MinMaxScaler()],  # Compare scalers
            'regressor': [Ridge(), Lasso(max_iter=5000), ElasticNet(max_iter=5000)],
            'regressor__alpha': [0.01, 0.1, 1, 10, 100, 1000]  
        }
        
        # Create pipeline
        pipe_radius = Pipeline(steps=[
            ('polyFeature', PolynomialFeatures()),
            ('scaler', StandardScaler()),
            ('regressor', Ridge())  # Default model
        ])
        
        X_train = globals()[f"X{i}_train"]
        X_test = globals()[f"X{i}_test"]
        
        # Use GridSearchCV with KFold for regression
        grid_search = GridSearchCV(pipe_radius, param_grid, cv=KFold(n_splits=5), scoring='r2')
        
        try:
            grid_search.fit(X_train, y_train)
        except Exception as e:
            print(f"Error during model fitting: {e}")
            continue
        
        # Best parameters and score
        best_params = grid_search.best_params_
        best_score = grid_search.best_score_
        print(f"Best Hyperparameters {n}, filter {i}: {best_params}")
        print(f"Best Score: {best_score:.4f}")
        
        # Predict using the best model
        try:
            y_predict = grid_search.best_estimator_.predict(X_test)
        except Exception as e:
            print(f"Error during prediction: {e}")
            continue
        
        # Evaluate model performance
        try:
            r2 = r2_score(y_test, y_predict)
        except Exception as e:
            print(f"Error during scoring: {e}")
            continue

        r_2_for_each_i.append(r2)
        print(f"R^2 Score for degree {n}: {r2:.4f}")
        
        # Plot results
        ax = axes[(n-1)//2, (n-1)%2]
        ax.scatter(y_predict, y_test, color='blue', alpha=0.5, label='Predicted vs True')
        ax.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Fit')
        ax.set_xlabel('Predicted Values')
        ax.set_ylabel('True Values')
        ax.set_title(f"Model with polyFeature = {n}")
        ax.legend()
    
    r_2_for_Rp_Re.append(r_2_for_each_i)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold

r_2_for_Rp_Re_rf = []

for i in range(1, 4):  # Best Filter Use
    if i == 1:
        print("Use Both Filter")
    elif i == 2:
        print("Use Bessel Filter")
    elif i == 3:
        print("Use Sloan Filter")
    
    r_2_for_each_i = []
    
    print("\nTraining Random Forest Model")
    
    # Define parameter grid for fine-tuning
    param_grid = {
        'scaler': [StandardScaler(), MinMaxScaler()],  # Compare scalers
        'regressor': [RandomForestRegressor()],  # Use Random Forest
        'regressor__n_estimators': [50, 100, 200],  # Tune number of trees
        'regressor__max_depth': [None, 10, 20, 30]  # Tune tree depth
    }
    
    # Create pipeline
    pipe_radius = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('regressor', RandomForestRegressor())  # Default model
    ])
    
    X_train = globals()[f"X{i}_train"]
    X_test = globals()[f"X{i}_test"]
    
    # Use GridSearchCV with KFold for regression
    grid_search = GridSearchCV(pipe_radius, param_grid, cv=KFold(n_splits=5), scoring='r2')
    
    try:
        grid_search.fit(X_train, y_train)
    except Exception as e:
        print(f"Error during model fitting: {e}")
        continue
    
    # Best parameters and score
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    print(f"Best Hyperparameters, filter {i}: {best_params}")
    print(f"Best Score: {best_score:.4f}")
    
    # Predict using the best model
    try:
        y_predict = grid_search.best_estimator_.predict(X_test)
    except Exception as e:
        print(f"Error during prediction: {e}")
        continue
    
    # Evaluate model performance
    try:
        r2 = r2_score(y_test, y_predict)
    except Exception as e:
        print(f"Error during scoring: {e}")
        continue

    r_2_for_each_i.append(r2)
    print(f"R^2 Score: {r2:.4f}")
    
    # Plot results
    plt.figure(figsize=(7, 5))
    plt.scatter(y_predict, y_test, color='blue', alpha=0.5, label='Predicted vs True')
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Fit')
    plt.xlabel('Predicted Values')
    plt.ylabel('True Values')
    plt.title("Random Forest Regression")
    plt.legend()
    plt.show()
    
    r_2_for_Rp_Re_rf.append(r_2_for_each_i)

plt.tight_layout()
plt.show()


## mass